In [1]:
!pip install Faker pandas numpy

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------------------------------------ --- 1.8/2.0 MB 17.8 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 13.7 MB/s  0:00:00


In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)
np.random.seed(42)
-
num_clientes = 1000
clientes_data = []

for cliente_id in range(1, num_clientes + 1):
    data_cadastro = fake.date_between(start_date='-3y', end_date='today')
    
    clientes_data.append({
        "id_cliente": cliente_id,
        "nome": fake.name(),
        "email": fake.email(),
        "cidade": fake.city(),
        "estado": fake.state_abbr(),
        "data_cadastro": pd.to_datetime(data_cadastro)
    })

df_clientes = pd.DataFrame(clientes_data)

num_vendas = 5000
vendas_data = []
canais_marketing = ['Google Ads', 'Meta Ads', 'E-mail Marketing', 'Orgânico', 'Influenciadores']

for venda_id in range(1, num_vendas + 1):
    
    cliente_aleatorio = df_clientes.sample(1).iloc[0]
    id_cliente = cliente_aleatorio['id_cliente']
    data_cad_cliente = cliente_aleatorio['data_cadastro']
    
    data_venda = fake.date_between(start_date=data_cad_cliente, end_date='today')
    
    valor_venda = round(np.random.exponential(scale=150.0) + 20, 2)
    
    canal = random.choice(canais_marketing)
    
    vendas_data.append({
        "id_venda": venda_id,
        "id_cliente": id_cliente,
        "data_venda": pd.to_datetime(data_venda),
        "valor": valor_venda,
        "canal_aquisicao": canal
    })

df_vendas = pd.DataFrame(vendas_data)

print("--- Amostra de Clientes ---")
display(df_clientes.head())
print("\n--- Amostra de Vendas ---")
display(df_vendas.head())

--- Amostra de Clientes ---


,id_cliente,nome,email,cidade,estado,data_cadastro
0,1,Apollo Sousa,samuel32@example.net,Rocha,SC,2026-03-05
1,2,Otávio Andrade,borgespedro-henrique@example.org,Novais,DF,2025-10-10
2,3,Dra. Mariah Câmara,rpastor@example.org,Sampaio,PA,2026-03-21
3,4,Eduardo Cunha,gustavo16@example.org,Ferreira Paulista,SE,2024-11-25
4,5,Sra. Zoe Campos,jbarros@example.com,Oliveira do Oeste,RR,2023-08-22



--- Amostra de Vendas ---


,id_venda,id_cliente,data_venda,valor,canal_aquisicao
0,1,522,2025-07-13,710.18,Google Ads
1,2,94,2025-04-11,91.55,Google Ads
2,3,613,2026-03-20,89.77,E-mail Marketing
3,4,556,2026-05-10,171.81,Meta Ads
4,5,677,2026-01-27,111.01,Meta Ads


In [3]:
import sqlite3

conn = sqlite3.connect('ecommerce.db')

df_clientes.to_sql('dim_clientes', conn, if_exists='replace', index=False)
df_vendas.to_sql('fato_vendas', conn, if_exists='replace', index=False)

print("Banco de dados 'ecommerce.db' criado e tabelas populadas com sucesso!")

conn.close()

Banco de dados 'ecommerce.db' criado e tabelas populadas com sucesso!


In [4]:
conn = sqlite3.connect('ecommerce.db')

query_visao_geral = """
SELECT 
    COUNT(id_venda) AS total_vendas,
    ROUND(SUM(valor), 2) AS faturamento_total,
    ROUND(AVG(valor), 2) AS ticket_medio
FROM fato_vendas;
"""

df_visao_geral = pd.read_sql_query(query_visao_geral, conn)
print("--- Indicadores Principais de Negócio ---")
display(df_visao_geral)
conn.close()

--- Indicadores Principais de Negócio ---


,total_vendas,faturamento_total,ticket_medio
0,5000,841648.0,168.33


In [5]:
conn = sqlite3.connect('ecommerce.db')

query_canais = """
SELECT 
    canal_aquisicao,
    COUNT(id_venda) AS quantidade_vendas,
    ROUND(SUM(valor), 2) AS faturamento_por_canal,
    ROUND(AVG(valor), 2) AS ticket_medio_por_canal
FROM fato_vendas
GROUP BY canal_aquisicao
ORDER BY faturamento_por_canal DESC;
"""

df_canais = pd.read_sql_query(query_canais, conn)
print("\n--- Desempenho por Canal de Marketing ---")
display(df_canais)
conn.close()


--- Desempenho por Canal de Marketing ---


,canal_aquisicao,quantidade_vendas,faturamento_por_canal,ticket_medio_por_canal
0,Orgânico,1016,174946.50,172.19
1,Influenciadores,1018,170659.18,167.64
2,E-mail Marketing,996,166156.57,166.82
3,Google Ads,985,164960.84,167.47
4,Meta Ads,985,164924.91,167.44


In [6]:
conn = sqlite3.connect('ecommerce.db')

query_estados = """
SELECT 
    c.estado,
    COUNT(v.id_venda) AS total_vendas,
    ROUND(SUM(v.valor), 2) AS faturamento_estado
FROM fato_vendas v
INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
GROUP BY c.estado
ORDER BY faturamento_estado DESC
LIMIT 5; -- Mostra apenas os 5 estados que mais faturam
"""

df_estados = pd.read_sql_query(query_estados, conn)
print("\n--- Top 5 Estados em Faturamento (Uso de JOIN) ---")
display(df_estados)
conn.close()


--- Top 5 Estados em Faturamento (Uso de JOIN) ---


,estado,total_vendas,faturamento_estado
0,SC,242,39976.61
1,PB,236,38018.87
2,PR,187,37536.30
3,BA,201,37222.86
4,GO,199,36615.30


In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ecommerce.db')

query_recorrencia = """
WITH historico_clientes AS (
    SELECT 
        id_cliente,
        COUNT(id_venda) AS total_compras,
        ROUND(SUM(valor), 2) AS total_gasto,
        ROUND(AVG(valor), 2) AS ticket_medio_cliente
    FROM fato_vendas
    GROUP BY id_cliente
)
SELECT 
    CASE 
        WHEN total_compras = 1 THEN '1. Comprou uma única vez'
        WHEN total_compras BETWEEN 2 AND 4 THEN '2. Cliente Recorrente (2-4 compras)'
        ELSE '3. Cliente Fiel / VIP (5+ compras)'
    END AS status_fidelidade,
    COUNT(id_cliente) AS quantidade_clientes,
    ROUND(SUM(total_gasto), 2) AS faturamento_acumulado,
    ROUND(AVG(ticket_medio_cliente), 2) AS ticket_medio_grupo
FROM historico_clientes
GROUP BY status_fidelidade
ORDER BY status_fidelidade;
"""

df_recorrencia = pd.read_sql_query(query_recorrencia, conn)
print("--- Análise de Recorrência de Clientes ---")
display(df_recorrencia)
conn.close()

--- Análise de Recorrência de Clientes ---


,status_fidelidade,quantidade_clientes,faturamento_acumulado,ticket_medio_grupo
0,1. Comprou uma única vez,36,7575.60,210.43
1,2. Cliente Recorrente (2-4 compras),383,211330.22,173.68
2,3. Cliente Fiel / VIP (5+ compras),572,622742.18,166.04


In [8]:
conn = sqlite3.connect('ecommerce.db')

query_churn_risco = """
WITH ultima_compra_por_cliente AS (
    SELECT 
        v.id_cliente,
        c.nome,
        c.email,
        MAX(v.data_venda) AS data_ultima_compra,
        SUM(v.valor) AS valor_total_historico,
        -- Buscando a data máxima global do banco para simular o 'hoje'
        (SELECT MAX(data_venda) FROM fato_vendas) AS data_atual_banco
    FROM fato_vendas v
    INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
    GROUP BY v.id_cliente
),
calculo_ociosidade AS (
    SELECT 
        id_cliente,
        nome,
        email,
        data_ultima_compra,
        valor_total_historico,
        -- Calcula a diferença em dias entre a última compra e a data atual do banco
        CAST(julianday(data_atual_banco) - julianday(data_ultima_compra) AS INT) AS dias_sem_comprar
    FROM ultima_compra_por_cliente
)
SELECT 
    id_cliente,
    nome,
    email,
    dias_sem_comprar,
    ROUND(valor_total_historico, 2) AS ltv_cliente,
    CASE 
        WHEN dias_sem_comprar <= 30 THEN 'Ativo (Comprou nos últimos 30 dias)'
        WHEN dias_sem_comprar BETWEEN 31 AND 90 THEN 'Alerta (Sem compras de 1 a 3 meses)'
        ELSE 'Churn / Inativo (Mais de 90 dias sem comprar)'
    END AS status_atividade
FROM calculo_ociosidade
WHERE status_activity LIKE 'Churn%' AND ltv_cliente > 200 -- Filtrando clientes valiosos que deram churn
ORDER BY ltv_cliente DESC
LIMIT 10;
"""

# Nota: Ajustando um pequeno detalhe do alias na query para o SQLite
query_churn_risco = query_churn_risco.replace("WHERE status_activity", "WHERE status_atividade")

df_churn = pd.read_sql_query(query_churn_risco, conn)
print("--- Top 10 Clientes de Alto Valor em Churn (Inativos há > 90 dias) ---")
display(df_churn)
conn.close()

--- Top 10 Clientes de Alto Valor em Churn (Inativos há > 90 dias) ---


,id_cliente,nome,email,dias_sem_comprar,ltv_cliente,status_atividade
0,42,Sra. Sofia das Neves,zduarte@example.net,267,3013.63,Churn / Inativo (Mais de 90 dias sem comprar)
1,10,Dra. Zoe da Luz,zpacheco@example.com,132,2913.12,Churn / Inativo (Mais de 90 dias sem comprar)
2,876,Yuri Siqueira,theogarcia@example.com,93,2540.29,Churn / Inativo (Mais de 90 dias sem comprar)
3,149,Bryan Teixeira,yagomachado@example.net,140,2514.31,Churn / Inativo (Mais de 90 dias sem comprar)
4,107,Arthur Miguel da Rosa,gcamargo@example.org,210,2464.92,Churn / Inativo (Mais de 90 dias sem comprar)
5,317,Otto Costa,yasminvieira@example.org,235,2353.71,Churn / Inativo (Mais de 90 dias sem comprar)
6,474,Levi Dias,vitor96@example.com,205,2304.64,Churn / Inativo (Mais de 90 dias sem comprar)
7,98,Manuella Pastor,lais47@example.com,454,2089.02,Churn / Inativo (Mais de 90 dias sem comprar)
8,991,Elisa Albuquerque,rlima@example.com,224,2076.70,Churn / Inativo (Mais de 90 dias sem comprar)
9,698,Maitê da Mata,nunessofia@example.com,159,1954.31,Churn / Inativo (Mais de 90 dias sem comprar)


In [9]:
!pip install streamlit plotly

In [10]:
%%writefile app.py
import streamlit as str
import pandas as pd
import sqlite3
import plotly.express as px


st.set_page_config(page_title="Dashboard Executivo de E-commerce", layout="wide")

st.title("📊 Dashboard Executivo de E-commerce")
st.markdown("Análise de vendas, canais de marketing e saúde da base de clientes.")


conn = sqlite3.connect('ecommerce.db')

query_kpi = "SELECT COUNT(id_venda) as qtd, SUM(valor) as fat, AVG(valor) as tk_medio FROM fato_vendas;"
df_kpi = pd.read_sql_query(query_kpi, conn)

query_canais = """
SELECT canal_aquisicao, SUM(valor) as faturamento 
FROM fato_vendas 
GROUP BY canal_aquisicao 
ORDER BY faturamento DESC;
"""
df_canais = pd.read_sql_query(query_canais, conn)


query_recorrencia = """
WITH hist AS (SELECT id_cliente, COUNT(id_venda) as compras FROM fato_vendas GROUP BY id_cliente)
SELECT 
    CASE 
        WHEN compras = 1 THEN '1. Uma única vez'
        WHEN compras BETWEEN 2 AND 4 THEN '2. Recorrente (2-4)'
        ELSE '3. VIP (5+)'
    END as status,
    COUNT(id_cliente) as total_clientes
FROM hist GROUP BY status;
"""
df_recorrencia = pd.read_sql_query(query_recorrencia, conn)
conn.close()

# --- LAYOUT DO STREAMLIT ---


col1, col2, col3 = st.columns(3)
col1.metric("Faturamento Total", f"R$ {df_kpi['fat'].iloc[0]:,.2f}")
col2.metric("Total de Vendas", f"{df_kpi['qtd'].iloc[0]:,}")
col3.metric("Ticket Médio", f"R$ {df_kpi['tk_medio'].iloc[0]:,.2f}")

st.markdown("---")


col_esq, col_dir = st.columns(2)

with col_esq:
    st.subheader("Faturamento por Canal de Marketing")
    fig_canais = px.bar(df_canais, x='canal_aquisicao', y='faturamento', 
                        labels={'canal_aquisicao': 'Canal', 'faturamento': 'Faturamento (R$)'},
                        color='faturamento', color_continuous_scale='Blues')
    st.plotly_chart(fig_canais, use_container_width=True)

with col_dir:
    st.subheader("Fidelidade da Base de Clientes")
    fig_rec = px.pie(df_recorrencia, values='total_clientes', names='status', 
                     hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
    st.plotly_chart(fig_rec, use_container_width=True)

Writing app.py


In [11]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

# Configuração da página do navegador
st.set_page_config(page_title="Dashboard Executivo de E-commerce", layout="wide")

st.title("📊 Dashboard Executivo de E-commerce")
st.markdown("Análise de vendas, canais de marketing e saúde da base de clientes.")

# Conexão com o banco de dados
conn = sqlite3.connect('ecommerce.db')

# --- 1. BUSCANDO OS DADOS DOS KPIs ---
query_kpi = "SELECT COUNT(id_venda) as qtd, SUM(valor) as fat, AVG(valor) as tk_medio FROM fato_vendas;"
df_kpi = pd.read_sql_query(query_kpi, conn)

# --- 2. BUSCANDO DADOS PARA O GRÁFICO DE CANAIS ---
query_canais = """
SELECT canal_aquisicao, SUM(valor) as faturamento 
FROM fato_vendas 
GROUP BY canal_aquisicao 
ORDER BY faturamento DESC;
"""
df_canais = pd.read_sql_query(query_canais, conn)

# --- 3. BUSCANDO DADOS DE RECORRÊNCIA ---
query_recorrencia = """
WITH hist AS (SELECT id_cliente, COUNT(id_venda) as compras FROM fato_vendas GROUP BY id_cliente)
SELECT 
    CASE 
        WHEN compras = 1 THEN '1. Uma única vez'
        WHEN compras BETWEEN 2 AND 4 THEN '2. Recorrente (2-4)'
        ELSE '3. VIP (5+)'
    END as status,
    COUNT(id_cliente) as total_clientes
FROM hist GROUP BY status;
"""
df_recorrencia = pd.read_sql_query(query_recorrencia, conn)
conn.close()

# --- LAYOUT DO STREAMLIT ---

# Linha 1: Cartões de KPIs (Métricas Principais)
col1, col2, col3 = st.columns(3)
col1.metric("Faturamento Total", f"R$ {df_kpi['fat'].iloc[0]:,.2f}")
col2.metric("Total de Vendas", f"{df_kpi['qtd'].iloc[0]:,}")
col3.metric("Ticket Médio", f"R$ {df_kpi['tk_medio'].iloc[0]:,.2f}")

st.markdown("---")

# Linha 2: Gráficos Lado a Lado
col_esq, col_dir = st.columns(2)

with col_esq:
    st.subheader("Faturamento por Canal de Marketing")
    fig_canais = px.bar(df_canais, x='canal_aquisicao', y='faturamento', 
                        labels={'canal_aquisicao': 'Canal', 'faturamento': 'Faturamento (R$)'},
                        color='faturamento', color_continuous_scale='Blues')
    st.plotly_chart(fig_canais, use_container_width=True)

with col_dir:
    st.subheader("Fidelidade da Base de Clientes")
    fig_rec = px.pie(df_recorrencia, values='total_clientes', names='status', 
                     hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
    st.plotly_chart(fig_rec, use_container_width=True)

Overwriting app.py


In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

st.set_page_config(page_title="Dashboard Executivo de E-commerce", layout="wide")

st.title("📊 Dashboard Executivo de E-commerce")
st.markdown("Análise de vendas, canais de marketing e saúde da base de clientes.")

conn = sqlite3.connect('ecommerce.db')

estados_disponiveis = pd.read_sql_query("SELECT DISTINCT estado FROM dim_clientes ORDER BY estado;", conn)['estado'].tolist()
canais_disponiveis = pd.read_sql_query("SELECT DISTINCT canal_aquisicao FROM fato_vendas ORDER BY canal_aquisicao;", conn)['canal_aquisicao'].tolist()

st.sidebar.header("Filtros do Dashboard")
estados_selecionados = st.sidebar.multiselect("Selecione os Estados:", estados_disponiveis, default=estados_disponiveis)
canais_selecionados = st.sidebar.multiselect("Selecione os Canais de Marketing:", canais_disponiveis, default=canais_disponiveis)

estados_sql = "', '".join(estados_selecionados)
canais_sql = "', '".join(canais_selecionados)

query_kpi = f"""
SELECT 
    COUNT(v.id_venda) as qtd, 
    SUM(v.valor) as fat, 
    AVG(v.valor) as tk_medio 
FROM fato_vendas v
INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}');
"""
df_kpi = pd.read_sql_query(query_kpi, conn)

query_canais = f"""
SELECT v.canal_aquisicao, SUM(v.valor) as faturamento 
FROM fato_vendas v
INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}')
GROUP BY v.canal_aquisicao 
ORDER BY faturamento DESC;
"""
df_canais = pd.read_sql_query(query_canais, conn)

query_recorrencia = f"""
WITH hist AS (
    SELECT v.id_cliente, COUNT(v.id_venda) as compras 
    FROM fato_vendas v
    INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
    WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}')
    GROUP BY v.id_cliente
)
SELECT 
    CASE 
        WHEN compras = 1 THEN '1. Uma única vez'
        WHEN compras BETWEEN 2 AND 4 THEN '2. Recorrente (2-4)'
        ELSE '3. VIP (5+)'
    END as status,
    COUNT(id_cliente) as total_clientes
FROM hist GROUP BY status;
"""
df_recorrencia = pd.read_sql_query(query_recorrencia, conn)
conn.close()

faturamento = df_kpi['fat'].iloc[0] if df_kpi['fat'].iloc[0] is not None else 0.0
total_vendas = df_kpi['qtd'].iloc[0] if df_kpi['qtd'].iloc[0] is not None else 0
ticket_medio = df_kpi['tk_medio'].iloc[0] if df_kpi['tk_medio'].iloc[0] is not None else 0.0

col1, col2, col3 = st.columns(3)
col1.metric("Faturamento Total", f"R$ {faturamento:,.2f}")
col2.metric("Total de Vendas", f"{total_vendas:,}")
col3.metric("Ticket Médio", f"R$ {ticket_medio:,.2f}")

st.markdown("---")

col_esq, col_dir = st.columns(2)

with col_esq:
    st.subheader("Faturamento por Canal de Marketing")
    if not df_canais.empty:
        fig_canais = px.bar(df_canais, x='canal_aquisicao', y='faturamento', 
                            labels={'canal_aquisicao': 'Canal', 'faturamento': 'Faturamento (R$)'},
                            color='faturamento', color_continuous_scale='Blues')
        st.plotly_chart(fig_canais, use_container_width=True)
    else:
        st.warning("Nenhum dado encontrado para os filtros selecionados.")

with col_dir:
    st.subheader("Fidelidade da Base de Clientes")
    if not df_recorrencia.empty:
        fig_rec = px.pie(df_recorrencia, values='total_clientes', names='status', 
                         hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
        st.plotly_chart(fig_rec, use_container_width=True)
    else:
        st.warning("Nenhum dado encontrado para os filtros selecionados.")

Overwriting app.py


In [13]:
!pip install scikit-learn

In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px
import numpy as np
from sklearn.linear_model import LinearRegression

st.set_page_config(page_title="Dashboard Executivo de E-commerce", layout="wide")

st.title("📊 Dashboard Executivo de E-commerce")
st.markdown("Análise de vendas, canais de marketing e saúde da base de clientes com inteligência preditiva.")

conn = sqlite3.connect('ecommerce.db')

estados_disponiveis = pd.read_sql_query("SELECT DISTINCT estado FROM dim_clientes ORDER BY estado;", conn)['estado'].tolist()
canais_disponiveis = pd.read_sql_query("SELECT DISTINCT canal_aquisicao FROM fato_vendas ORDER BY canal_aquisicao;", conn)['canal_aquisicao'].tolist()

st.sidebar.header("Filtros do Dashboard")
estados_selecionados = st.sidebar.multiselect("Selecione os Estados:", estados_disponiveis, default=estados_disponiveis)
canais_selecionados = st.sidebar.multiselect("Selecione os Canais de Marketing:", canais_disponiveis, default=canais_disponiveis)

estados_sql = "', '".join(estados_selecionados)
canais_sql = "', '".join(canais_selecionados)

query_kpi = f"""
SELECT 
    COUNT(v.id_venda) as qtd, 
    SUM(v.valor) as fat, 
    AVG(v.valor) as tk_medio 
FROM fato_vendas v
INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}');
"""
df_kpi = pd.read_sql_query(query_kpi, conn)

query_canais = f"""
SELECT v.canal_aquisicao, SUM(v.valor) as faturamento 
FROM fato_vendas v
INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}')
GROUP BY v.canal_aquisicao 
ORDER BY faturamento DESC;
"""
df_canais = pd.read_sql_query(query_canais, conn)

query_recorrencia = f"""
WITH hist AS (
    SELECT v.id_cliente, COUNT(v.id_venda) as compras 
    FROM fato_vendas v
    INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
    WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}')
    GROUP BY v.id_cliente
)
SELECT 
    CASE 
        WHEN compras = 1 THEN '1. Uma única vez'
        WHEN compras BETWEEN 2 AND 4 THEN '2. Recorrente (2-4)'
        ELSE '3. VIP (5+)'
    END as status,
    COUNT(id_cliente) as total_clientes
FROM hist GROUP BY status;
"""
df_recorrencia = pd.read_sql_query(query_recorrencia, conn)

query_vendas_tempo = f"""
SELECT v.data_venda, v.valor
FROM fato_vendas v
INNER JOIN dim_clientes c ON v.id_cliente = c.id_cliente
WHERE c.estado IN ('{estados_sql}') AND v.canal_aquisicao IN ('{canais_sql}');
"""
df_tempo = pd.read_sql_query(query_vendas_tempo, conn)
conn.close()

faturamento = df_kpi['fat'].iloc[0] if df_kpi['fat'].iloc[0] is not None else 0.0
total_vendas = df_kpi['qtd'].iloc[0] if df_kpi['qtd'].iloc[0] is not None else 0
ticket_medio = df_kpi['tk_medio'].iloc[0] if df_kpi['tk_medio'].iloc[0] is not None else 0.0

col1, col2, col3 = st.columns(3)
col1.metric("Faturamento Total", f"R$ {faturamento:,.2f}")
col2.metric("Total de Vendas", f"{total_vendas:,}")
col3.metric("Ticket Médio", f"R$ {ticket_medio:,.2f}")

st.markdown("---")

st.subheader("🔮 Previsão de Faturamento Próximos Meses (Machine Learning)")
if not df_tempo.empty:
    df_tempo['data_venda'] = pd.to_datetime(df_tempo['data_venda'])
    df_mensal = df_tempo.set_index('data_venda').resample('ME')['valor'].sum().reset_index()
    df_mensal.columns = ['Mes', 'Faturamento']
    df_mensal = df_mensal.sort_values('Mes').reset_index(drop=True)
    
    if len(df_mensal) > 2:
        df_mensal['Mes_Num'] = df_mensal.index + 1
        
        X = df_mensal[['Mes_Num']]
        y = df_mensal['Faturamento']
        
        modelo = LinearRegression()
        modelo.fit(X, y)
        
        ult_mes_num = df_mensal['Mes_Num'].max()
        ult_data = df_mensal['Mes'].max()
        
        futuro_meses = [ult_mes_num + 1, ult_mes_num + 2, ult_mes_num + 3]
        datas_futuras = [ult_data + pd.DateOffset(months=i) for i in range(1, 4)]
        
        previsoes = modelo.predict(pd.DataFrame(futuro_meses, columns=['Mes_Num']))
        
        df_futuro = pd.DataFrame({'Mes': datas_futuras, 'Faturamento': previsoes})
        
        df_mensal['Tipo'] = 'Histórico Real'
        df_futuro['Tipo'] = 'Previsão (ML)'
        
        df_ultimo_ponto = df_mensal.tail(1).copy()
        df_ultimo_ponto['Tipo'] = 'Previsão (ML)'
        df_final = pd.concat([df_mensal, df_ultimo_ponto, df_futuro], ignore_index=True)
        
        fig_prev = px.line(df_final, x='Mes', y='Faturamento', color='Tipo',
                           labels={'Mes': 'Mês', 'Faturamento': 'Faturamento (R$)'},
                           color_discrete_map={'Histórico Real': '#1f77b4', 'Previsão (ML)': '#ff7f0e'})
        st.plotly_chart(fig_prev, use_container_width=True)
    else:
        st.info("Dados históricos insuficientes para traçar uma linha de tendência preditiva.")
else:
    st.warning("Nenhum dado encontrado para gerar previsões.")

st.markdown("---")

col_esq, col_dir = st.columns(2)

with col_esq:
    st.subheader("Faturamento por Canal de Marketing")
    if not df_canais.empty:
        fig_canais = px.bar(df_canais, x='canal_aquisicao', y='faturamento', 
                            labels={'canal_aquisicao': 'Canal', 'faturamento': 'Faturamento (R$)'},
                            color='faturamento', color_continuous_scale='Blues')
        st.plotly_chart(fig_canais, use_container_width=True)

with col_dir:
    st.subheader("Fidelidade da Base de Clientes")
    if not df_recorrencia.empty:
        fig_rec = px.pie(df_recorrencia, values='total_clientes', names='status', 
                         hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
        st.plotly_chart(fig_rec, use_container_width=True)

Overwriting app.py


In [15]:
%%writefile requirements.txt
streamlit
pandas
numpy
faker
plotly
scikit-learn

Writing requirements.txt


In [16]:
%%writefile README.md
# 📊 Pipeline de Dados End-to-End & Dashboard Preditivo para E-commerce

Este projeto simula o ecossistema de dados de uma empresa de e-commerce de grande porte, cobrindo desde a ingestão e modelagem de dados relacionais até a criação de uma camada preditiva de Machine Learning e visualização de KPIs de negócio em um Web App interativo.

## 🛠️ Tecnologias e Ferramentas Utilizadas
* **Linguagem Principal:** Python 3
* **Ingestão e Simulação:** Pandas, NumPy e Faker
* **Armazenamento e Modelagem:** SQL (SQLite) utilizando o modelo Star Schema
* **Inteligência Preditiva:** Scikit-Learn (Regressão Linear para Forecasting de faturamento)
* **Visualização de Dados:** Streamlit e Plotly

Writing README.md
